# Insights Lane - Testing Notebook

Checks the **anomaly model** (`src/anomaly.py`) and the **Investigator agent**
(`src/agents/investigator.py`).

- The model (Isolation Forest + stats) finds unusual monthly movements - no LLM.
- The Investigator is a tool-using agent: it scans, then drills into findings with its tools,
  then summarizes. It makes LLM calls.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

## Part 1 - The anomaly model (no API calls)

`detect_anomalies` flags unusual monthly movements, portfolio-wide or scoped. It only reports
movements that are both statistically off and practically large.

In [2]:
from src.anomaly import detect_anomalies

print("Portfolio:")
for a in detect_anomalies():
    print("  ", a["category"], a["month"], a["value"], "|", a["note"])

print("\nBuilding 17 (steady - expect nothing):", detect_anomalies(property="Building 17") or "nothing material")

Portfolio:


   real_estate_taxes 2024-M07 1086.0 | sign flip - 1086.0 vs its usual -4170.44
   other_general_expenses 2024-M03 1581.94 | sign flip - 1581.94 vs its usual -364.85
   rent_discount_untaxed 2024-M02 -332.26 | well above its usual -1315.46
   rent_discount_untaxed 2024-M01 -14874.04 | well below its usual -1315.46
   other_consultancy_costs 2025-M03 -9300.0 | well below its usual -1025.06
   insurance_in_general 2024-M10 -817.49 | well above its usual -2534.84

Building 17 (steady - expect nothing): nothing material


## Part 2 - The drill-down tools (no API calls)

The agent uses these to understand a flagged point: the category's history over time, and who
drove a given month.

In [3]:
from src.anomaly import monthly_series, contributors

print("rent_discount_untaxed history:")
for r in monthly_series("rent_discount_untaxed"):
    print("  ", r)

print("\nWho drove rent_discount_untaxed in 2024-M01:")
for r in contributors("rent_discount_untaxed", "2024-M01"):
    print("  ", r)

rent_discount_untaxed history:
   {'month': '2024-M01', 'value': -14874.04}
   {'month': '2024-M02', 'value': -332.26}
   {'month': '2024-M03', 'value': -355.17}
   {'month': '2024-M04', 'value': -343.71}
   {'month': '2024-M05', 'value': -355.17}
   {'month': '2024-M06', 'value': -343.72}
   {'month': '2024-M07', 'value': -351.31}
   {'month': '2024-M08', 'value': -351.31}
   {'month': '2024-M09', 'value': -339.98}
   {'month': '2024-M10', 'value': -347.53}
   {'month': '2024-M11', 'value': -347.53}
   {'month': '2024-M12', 'value': -347.54}
   {'month': '2025-M01', 'value': -347.53}
   {'month': '2025-M02', 'value': -347.53}
   {'month': '2025-M03', 'value': -347.54}

Who drove rent_discount_untaxed in 2024-M01:
   {'who': 'Tenant 3', 'value': -14518.87}
   {'who': 'Tenant 2', 'value': -355.17}


## Part 3 - The Investigator agent (makes API calls)

`investigate` runs the tool loop. It returns the final answer and which tools it chose - the
tool list shows it isn't a fixed script; it follows the findings.

In [4]:
from src.agents.investigator import investigate

r = investigate("Is anything unusual? Look into the biggest one.")
print("tools used:", r["tools_used"])
print()
print(r["answer"])

tools used: ['find_anomalies', 'who_drove', 'category_history']

## Key Finding

**Rent discount spike in January 2024**: A one-time anomaly of **-14,874.04** (vs. typical **-1,315.46**), driven almost entirely by **Tenant 3** at **-14,518.87**. This appears to be a one-off event — discounts returned to normal levels (~-350) from February onwards and have remained stable since. 

Worth investigating whether this was a legitimate one-time concession (e.g., move-in incentive, lease renegotiation) or a data entry error, but it doesn't appear to be an ongoing issue.


## Part 4 - End to end through the graph

The router should send an "is anything unusual" question to the insights lane.

In [5]:
from src.graph import ask

r = ask("anything weird going on with the numbers?", thread_id="insights-nb")
print("intent:", r["intent"])
print("reasoning:", r["reasoning"])
print()
print(r["answer"][:600])

intent: insights
reasoning: Investigated with tools: find_anomalies, category_history, category_history, category_history, who_drove, who_drove

## Summary of Notable Findings

**1. Rent Discount Spike (January 2024): –$14,874**
   - Typical monthly rent discounts run around –$1,315, but January 2024 spiked to –$14,874—over 11× normal.
   - **Tenant 3 drove this**, accounting for –$14,519 of the discount in that month alone.
   - This appears to be a one-off event; discounts normalized to ~–$340–350/month from February onward.

**2. Consultancy Costs Spike (March 2025): –$9,300**
   - Typical monthly consultancy costs are around –$1,025, but March 2025 jumped to –$9,300—9× normal.
   - This is a sharp departure from the ~–$250 baseline


## Notes

- **The model does the maths** (Isolation Forest + statistics); the LLM only decides which tool
  to call and phrases the result - so the figures stay deterministic.
- **The tool list in Part 3** shows genuine choice (e.g. `find_anomalies` then `who_drove` then
  `category_history`), not a fixed sequence.
- The analytics path stays the controlled SQL pipeline; the tool-using agent lives only here,
  where investigation is open-ended.